In [2]:
# AG System with Access Control + JSON Enforcement + Validation
import json
from jsonschema import validate,ValidationError

In [3]:
Restricted_Keywords = ["salary","compensation","bonus","payroll"]
Allowed_Keywords =["policy","leave","holiday","benefits"]

def classify_query(query:str):
    query=query.lower()
    for word in Restricted_Keywords:
        if word in query:
            return "restricted"
    for word in Allowed_Keywords:
        if word in query:
            return "allowed"
    return "unknown"

def access_control(query):
    classification = classify_query(query)

    if classification =="restricted":
        return False,"Access Denied: Restricted Information"
    return True, "Access Granted"


DOCUMENTS = [
    "Company leave policy allows 20 days annual leave.",
    "Office holidays are declared annually.",
    "Employees receive health insurance benefits."
]

def retriever(query):
    # Simple retriever (for now returns all documents)
    return " ".join(DOCUMENTS)


In [5]:
Response_Schema ={
    "type" :"object",
    "properties":{
        "answer": {"type":"string"},
        "confidence":{"type":"number"},
        "source_used":{"type":"boolean"}
    },
    "required": ["answer","confidence","source_used"]
}


In [6]:
def build_prompt(context, query):
    return f"""
You are a company assistant.

Answer ONLY in valid JSON format.

Required schema:
{{
    "answer": "...",
    "confidence": float between 0 and 1,
    "source_used": boolean
}}

Context:
{context}

Question:
{query}

Return JSON only.
"""


In [7]:
def dummy_llm(prompt):
    return """
    {
        "answer": "Company allows 20 days annual leave.",
        "confidence": 0.88,
        "source_used": true
    }
    """


In [14]:
def validate_response(response_text):
    try:
        data = json.loads(response_text)
        validate(instance=data, schema=Response_Schema)
        return True, data
    except (json.JSONDecodeError, ValidationError):
        return False, None


In [9]:
def generate_with_retry(llm_call, max_retries=3):
    for attempt in range(max_retries):
        response = llm_call()
        is_valid, parsed = validate_response(response)
        
        if is_valid:
            print(f"Valid JSON on attempt {attempt+1}")
            return parsed
        
        print(f"Retrying... Attempt {attempt+1}")
    
    raise ValueError("Failed to produce valid JSON after retries")


In [10]:
def hallucination_check(answer, context):
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())
    
    overlap = len(answer_words.intersection(context_words))
    
    return overlap >= 5


In [11]:
def validation_layer(response_json, context):
    answer = response_json["answer"]
    
    if not hallucination_check(answer, context):
        return False, "Rejected: Possible Hallucination"
    
    return True, response_json


In [15]:
def rag_pipeline(query, retriever, llm):
    
    # 1️⃣ Access Control
    allowed, message = access_control(query)
    if not allowed:
        return {"error": message}
    
    # 2️⃣ Retrieval
    context = retriever(query)
    
    # 3️⃣ Build Prompt
    prompt = build_prompt(context, query)
    
    # 4️⃣ LLM Call with Retry
    def llm_call():
        return llm(prompt)
    
    response_json = generate_with_retry(llm_call)
    
    # 5️⃣ Validation Layer
    valid, final_output = validation_layer(response_json, context)
    
    if not valid:
        return {"error": final_output}
    
    return final_output


In [16]:
rag_pipeline("What is company leave policy?", retriever, dummy_llm)


Valid JSON on attempt 1


{'answer': 'Company allows 20 days annual leave.',
 'confidence': 0.88,
 'source_used': True}

In [17]:
rag_pipeline("What is CEO salary?", retriever, dummy_llm)


{'error': 'Access Denied: Restricted Information'}

In [18]:
def dummy_llm_bad(prompt):
    return """
    {
        "answer": "CEO earns 10 million dollars.",
        "confidence": 0.95,
        "source_used": true
    }
    """
rag_pipeline("What is company leave policy?", retriever, dummy_llm_bad)



Valid JSON on attempt 1


{'error': 'Rejected: Possible Hallucination'}